In [1]:
pip install lime scikit-learn pandas numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 392.2 kB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=7a3d90bedfffecf3a15fbca1968eb38d4b0ec4a1e50abb928d064ebdd3009964
  Stored in directory: /home/jovyan/.cache/pip/wheels/85/fa/a3/9c2d44c9f3cd77cf4e533b58900b2bf4487f2a17e8ec212a3d
Successfully built lime
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import lime
import lime.lime_tabular



In [3]:
# Dataset Simulation : 

#For this lab, we'll simulate a dataset. In a real scenario, you would load a pre-existing dataset (e.g., from CIC-IDS2018 or a custom phishing dataset).

In [6]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import lime
import lime.lime_tabular
import os

print("If you see this message → your lab is working perfectly!")

# --- Configuration ---
# Use absolute paths relative to the student's home directory in the lab
#DATA_DIR = '/home/student/cyberai-lab/data'
#MODEL_DIR = '/home/student/cyberai-lab/models'

# --- Configuration ---
# OLD: DATA_DIR = '/home/student/cyberai-lab/data'
# NEW: Using relative paths so it works in your current folder
DATA_DIR = './data'
MODEL_DIR = './models'

# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# Create directories if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# --- Simulate Phishing Dataset ---
# In a real scenario, you would load a dataset like:
# df = pd.read_csv(os.path.join(DATA_DIR, 'phishing_emails.csv'))

np.random.seed(42)
num_samples = 1000

# Features simulating email characteristics
# Example features: number of links, presence of urgent keywords, sender domain reputation, etc.
features = {
    'num_links': np.random.randint(0, 10, num_samples),
    'has_urgent_keyword': np.random.randint(0, 2, num_samples),
    'sender_domain_reputation': np.random.uniform(0.1, 1.0, num_samples),
    'email_length': np.random.randint(50, 1000, num_samples),
    'contains_attachment': np.random.randint(0, 2, num_samples),
    'suspicious_url_pattern': np.random.randint(0, 2, num_samples)
}
df = pd.DataFrame(features)

# Simulate a target variable (1: phishing, 0: legitimate)
# Phishing emails are more likely to have more links, urgent keywords, lower domain reputation, etc.
phishing_prob = (
    0.1  # Base probability
    + df['num_links'] * 0.05
    + df['has_urgent_keyword'] * 0.15
    - df['sender_domain_reputation'] * 0.2
    + df['email_length'] * 0.0001
    + df['suspicious_url_pattern'] * 0.2
)
phishing_prob = np.clip(phishing_prob, 0.05, 0.95) # Clip probabilities
df['is_phishing'] = (np.random.rand(num_samples) < phishing_prob).astype(int)

print(f"Simulated dataset shape: {df.shape}")
print("Dataset head:")
print(df.head())
print("\nTarget distribution:")
print(df['is_phishing'].value_counts())

# --- Prepare Data for Model ---
X = df.drop('is_phishing', axis=1)
y = df['is_phishing']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# --- Train a Simple Classifier (Logistic Regression) ---
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nModel Accuracy on Test Set: {accuracy:.4f}")

# Expected output: Model Accuracy on Test Set: approximately 0.85-0.90 (will vary slightly due to random simulation)
print("Expected output: Model Accuracy on Test Set > 0.85")

# Save the model and data for later use if needed
# model_path = os.path.join(MODEL_DIR, 'phishing_logistic_regression.pkl')
# pd.to_pickle(model, model_path)
# X_test.to_csv(os.path.join(DATA_DIR, 'X_test_phishing.csv'), index=False)
# X.columns.to_series().to_csv(os.path.join(DATA_DIR, 'feature_names.csv'), index=False, header=False)

print("\nLab setup complete. Model trained and ready for LIME explanation.")

If you see this message → your lab is working perfectly!
Simulated dataset shape: (1000, 7)
Dataset head:
   num_links  has_urgent_keyword  sender_domain_reputation  email_length  \
0          6                   0                  0.723875           750   
1          3                   0                  0.839297            98   
2          7                   1                  0.137065           351   
3          4                   1                  0.703347           930   
4          6                   0                  0.956395           239   

   contains_attachment  suspicious_url_pattern  is_phishing  
0                    0                       0            0  
1                    0                       0            0  
2                    0                       0            1  
3                    0                       1            1  
4                    1                       1            0  

Target distribution:
is_phishing
0    539
1    461
Name: count, 

In [ ]:
# Applying LIME to Explain a Prediction



In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import lime
import lime.lime_tabular
import os

# --- Load previously saved model and data (or re-run setup if needed) ---
# For demonstration, we'll re-run the setup code to ensure consistency.
# In a real workflow, you might load saved artifacts.

# --- Simulate Phishing Dataset ---
np.random.seed(42)
num_samples = 1000
features = {
    'num_links': np.random.randint(0, 10, num_samples),
    'has_urgent_keyword': np.random.randint(0, 2, num_samples),
    'sender_domain_reputation': np.random.uniform(0.1, 1.0, num_samples),
    'email_length': np.random.randint(50, 1000, num_samples),
    'contains_attachment': np.random.randint(0, 2, num_samples),
    'suspicious_url_pattern': np.random.randint(0, 2, num_samples)
}
df = pd.DataFrame(features)
phishing_prob = (
    0.1 + df['num_links'] * 0.05 + df['has_urgent_keyword'] * 0.15 -
    df['sender_domain_reputation'] * 0.2 + df['email_length'] * 0.0001 +
    df['suspicious_url_pattern'] * 0.2
)
phishing_prob = np.clip(phishing_prob, 0.05, 0.95)
df['is_phishing'] = (np.random.rand(num_samples) < phishing_prob).astype(int)

X = df.drop('is_phishing', axis=1)
y = df['is_phishing']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- Train a Simple Classifier (Logistic Regression) ---
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# --- Select an instance to explain ---
# Let's pick the first instance from the test set
instance_idx = 0
instance_to_explain = X_test.iloc[[instance_idx]]

# Get the model's prediction for this instance
prediction = model.predict(instance_to_explain)[0]
prediction_proba = model.predict_proba(instance_to_explain)[0]

print(f"Explaining instance index: {instance_idx}")
print(f"Actual label: {y_test.iloc[instance_idx]}")
print(f"Predicted class: {prediction}")
print(f"Prediction probabilities: Phishing={prediction_proba[1]:.4f}, Legitimate={prediction_proba[0]:.4f}")

# --- Initialize LIME Tabular Explainer ---
# We need the training data (or a representative sample) to understand feature distributions
# and the feature names.
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['Legitimate', 'Phishing'],
    mode='classification'
)

# --- Generate the explanation ---
# The explanation is generated for a specific instance and the prediction function of the model.
# num_features: the number of features to include in the explanation.
# num_samples: the number of perturbed samples to generate for approximation.
explanation = explainer.explain_instance(
    data_row=instance_to_explain.iloc[0].values,
    predict_fn=model.predict_proba,
    num_features=6,  # Explain using all available features
    num_samples=500  # Number of perturbed samples
)

print("\nLIME explanation generated.")

# --- Visualize the explanation ---
# This will display a plot in the Jupyter Notebook output.
# The plot shows features contributing to the predicted class.
print("\nDisplaying LIME explanation visualization...")

# Get the explanation as a list of (feature, weight) tuples
# This is useful for programmatic analysis or custom visualizations
explanation_list = explanation.as_list()
print("\nExplanation as a list:")
for feature, weight in explanation_list:
    print(f"- {feature}: {weight:.4f}")

# The explanation.show_in_notebook(show_table=True, show_all=False) command
# is typically used in Jupyter notebooks to render the interactive visualization.
# For this JSON output, we'll describe the expected visualization.

print("\nExpected Visualization Output:\n")
print("A bar chart will be displayed. Features that positively contributed to the predicted class (e.g., 'Phishing') will be shown in one color (e.g., blue), and features that negatively contributed (i.e., supported the 'Legitimate' class) will be shown in another color (e.g., red). The length of the bar indicates the magnitude of the contribution.")
print("For example, if the model predicted 'Phishing', you might see features like 'has_urgent_keyword=1' or 'num_links=8' with positive weights, and 'sender_domain_reputation=0.8' with a negative weight.")

# --- Analyze the features that contributed most ---
print("\nAnalysis of Top Contributing Features:")
# The explanation.as_list() provides this information directly.
# We can sort it to explicitly show the top contributors.
top_contributors = sorted(explanation_list, key=lambda item: abs(item[1]), reverse=True)

print("Top 3 features contributing to the prediction (absolute weight):")
for i in range(min(3, len(top_contributors))):
    feature, weight = top_contributors[i]
    print(f"{i+1}. {feature} (Weight: {weight:.4f})")

print("\nThis analysis helps identify the specific characteristics of the instance that drove the model's decision.")

# Expected output: A list of features and their weights, showing which ones had the most impact.
print("Expected output: A list of features and their weights, indicating their influence on the prediction.")

Explaining instance index: 0
Actual label: 1
Predicted class: 0
Prediction probabilities: Phishing=0.3813, Legitimate=0.6187

LIME explanation generated.

Displaying LIME explanation visualization...

Explanation as a list:
- suspicious_url_pattern <= 0.00: -0.1721
- 0.00 < has_urgent_keyword <= 1.00: 0.1493
- sender_domain_reputation > 0.77: -0.1236
- 2.00 < num_links <= 4.00: -0.0704
- email_length > 769.25: 0.0353
- 0.00 < contains_attachment <= 1.00: -0.0294

Expected Visualization Output:

A bar chart will be displayed. Features that positively contributed to the predicted class (e.g., 'Phishing') will be shown in one color (e.g., blue), and features that negatively contributed (i.e., supported the 'Legitimate' class) will be shown in another color (e.g., red). The length of the bar indicates the magnitude of the contribution.
For example, if the model predicted 'Phishing', you might see features like 'has_urgent_keyword=1' or 'num_links=8' with positive weights, and 'sender_domai

/opt/conda/lib/python3.11/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [ ]:
#Analyzing Feature Contributions for Phishing Prediction

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import lime
import lime.lime_tabular
import os

# --- Re-run setup for consistency ---
np.random.seed(42)
num_samples = 1000
features = {
    'num_links': np.random.randint(0, 10, num_samples),
    'has_urgent_keyword': np.random.randint(0, 2, num_samples),
    'sender_domain_reputation': np.random.uniform(0.1, 1.0, num_samples),
    'email_length': np.random.randint(50, 1000, num_samples),
    'contains_attachment': np.random.randint(0, 2, num_samples),
    'suspicious_url_pattern': np.random.randint(0, 2, num_samples)
}
df = pd.DataFrame(features)
phishing_prob = (
    0.1 + df['num_links'] * 0.05 + df['has_urgent_keyword'] * 0.15 -
    df['sender_domain_reputation'] * 0.2 + df['email_length'] * 0.0001 +
    df['suspicious_url_pattern'] * 0.2
)
phishing_prob = np.clip(phishing_prob, 0.05, 0.95)
df['is_phishing'] = (np.random.rand(num_samples) < phishing_prob).astype(int)

X = df.drop('is_phishing', axis=1)
y = df['is_phishing']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# --- Select an instance to explain ---
instance_idx = 0
instance_to_explain = X_test.iloc[[instance_idx]]

# --- Initialize LIME Tabular Explainer ---
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['Legitimate', 'Phishing'],
    mode='classification'
)

# --- Generate the explanation ---
explanation = explainer.explain_instance(
    data_row=instance_to_explain.iloc[0].values,
    predict_fn=model.predict_proba,
    num_features=6,
    num_samples=500
)

# --- Programmatic Analysis of Feature Contributions ---
print("\n--- Analysis of Top Contributing Features ---")

# Get the explanation as a list of (feature_description, weight) tuples
explanation_list = explanation.as_list()

# Sort features by the absolute value of their weights to find the most impactful ones
# This helps in identifying features that strongly influenced the prediction, regardless of direction.
sorted_explanation = sorted(explanation_list, key=lambda item: abs(item[1]), reverse=True)

print(f"\nTop {min(5, len(sorted_explanation))} features contributing to the prediction for instance {instance_idx}:")

for i in range(min(5, len(sorted_explanation))):
    feature_desc, weight = sorted_explanation[i]
    # Determine if the contribution is positive or negative relative to the predicted class
    contribution_type = "positively" if weight > 0 else "negatively"
    print(f"{i+1}. Feature: '{feature_desc}', Weight: {weight:.4f} (contributed {contribution_type} to the predicted class)")

# --- Detailed breakdown of positive and negative contributions ---
print("\n--- Breakdown by Contribution Type ---")

positive_contributors = [(f, w) for f, w in explanation_list if w > 0]
negative_contributors = [(f, w) for f, w in explanation_list if w < 0]

# Sort positive contributors by weight (descending)
positive_contributors.sort(key=lambda item: item[1], reverse=True)
# Sort negative contributors by weight (ascending, so most negative first)
negative_contributors.sort(key=lambda item: item[1])

print(f"\nTop 3 features positively contributing to the predicted class:")
for i in range(min(3, len(positive_contributors))):
    feature_desc, weight = positive_contributors[i]
    print(f"{i+1}. '{feature_desc}' (Weight: {weight:.4f})")

print(f"\nTop 3 features negatively contributing to the predicted class (i.e., supporting the alternative class):")
for i in range(min(3, len(negative_contributors))):
    feature_desc, weight = negative_contributors[i]
    print(f"{i+1}. '{feature_desc}' (Weight: {weight:.4f})")

print("\nThis detailed analysis highlights the specific factors that swayed the model's decision for this particular instance.")

# Expected output: A structured list of features, their weights, and their contribution type (positive/negative).
print("Expected output: Structured lists of top positive and negative contributing features with their weights.")


--- Analysis of Top Contributing Features ---

Top 5 features contributing to the prediction for instance 0:
1. Feature: 'suspicious_url_pattern <= 0.00', Weight: -0.1721 (contributed negatively to the predicted class)
2. Feature: '0.00 < has_urgent_keyword <= 1.00', Weight: 0.1493 (contributed positively to the predicted class)
3. Feature: 'sender_domain_reputation > 0.77', Weight: -0.1236 (contributed negatively to the predicted class)
4. Feature: '2.00 < num_links <= 4.00', Weight: -0.0704 (contributed negatively to the predicted class)
5. Feature: 'email_length > 769.25', Weight: 0.0353 (contributed positively to the predicted class)

--- Breakdown by Contribution Type ---

Top 3 features positively contributing to the predicted class:
1. '0.00 < has_urgent_keyword <= 1.00' (Weight: 0.1493)
2. 'email_length > 769.25' (Weight: 0.0353)

Top 3 features negatively contributing to the predicted class (i.e., supporting the alternative class):
1. 'suspicious_url_pattern <= 0.00' (Weight:

/opt/conda/lib/python3.11/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
